In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import DecimalType
from pathlib import Path
import re

warehouse_location = "hdfs://hdfs-nn:9000/warehouse"

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("World Data 2017 - Bronze para Silver")
    .config("spark.sql.warehouse.dir", warehouse_location)
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-core_2.12:2.4.0")
    .enableHiveSupport()
    .getOrCreate()
)
print("Spark pronto:", spark.version)

Spark pronto: 3.4.1


In [3]:
import os

print("Pasta atual:", os.getcwd())

print("\nFicheiros CSV disponíveis:")
for ficheiro in os.listdir("/home/jovyan"):
    if ficheiro.lower().endswith(".csv"):
        print(ficheiro)

Pasta atual: /home/jovyan/work

Ficheiros CSV disponíveis:
global_superstore_orders_raw.csv
world-data-2023.csv


In [4]:
CSV_LOCAL = "/home/jovyan/work/world-data-2017_PIB_desemprego_CO2.csv"
RAW_DIR = "hdfs://hdfs-nn:9000/projeto/raw/world_data_2017"
CSV_HDFS = RAW_DIR + "/world-data-2017_PIB_desemprego_CO2.csv"
BRONZE_PATH = "hdfs://hdfs-nn:9000/projeto/bronze/world_data_2017"
SILVER_PATH = "hdfs://hdfs-nn:9000/warehouse/silver.db/world_data_2017_silver"
TABELA_HIVE = "silver.world_data_2017_silver"
HTML_PATH = "/home/jovyan/relatorio_data_profiling_world_data_2017_silver.html"

assert Path(CSV_LOCAL).is_file(), f"CSV não encontrado: {CSV_LOCAL}"
print("CSV encontrado:", CSV_LOCAL)
print("Bronze:", BRONZE_PATH)
print("Silver:", SILVER_PATH)
print("Hive:", TABELA_HIVE)

CSV encontrado: /home/jovyan/work/world-data-2017_PIB_desemprego_CO2.csv
Bronze: hdfs://hdfs-nn:9000/projeto/bronze/world_data_2017
Silver: hdfs://hdfs-nn:9000/warehouse/silver.db/world_data_2017_silver
Hive: silver.world_data_2017_silver


In [5]:
conf = spark.sparkContext._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
pasta = JPath(RAW_DIR)
fs = pasta.getFileSystem(conf)
fs.mkdirs(pasta)

origem = JPath(CSV_LOCAL)
destino = JPath(CSV_HDFS)
fs.copyFromLocalFile(False, True, origem, destino)

assert fs.exists(destino), "O CSV não foi copiado para HDFS."
print("CSV guardado no HDFS:", CSV_HDFS)

CSV guardado no HDFS: hdfs://hdfs-nn:9000/projeto/raw/world_data_2017/world-data-2017_PIB_desemprego_CO2.csv


In [7]:
def to_snake(nome):
    nome = nome.replace("\ufeff", "").strip().lower()
    return re.sub(r"[^a-z0-9]+", "_", nome).strip("_")

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .option("escape", '"')
    .csv(CSV_HDFS)
)

nomes = [to_snake(c) for c in raw.columns]
assert len(nomes) == len(set(nomes)), "Existem nomes duplicados após normalização."
raw = raw.toDF(*nomes)

print("Registos:", raw.count())
print("Colunas:", len(raw.columns))
raw.select("country", "year", "unemployment_rate", "gdp", "co2_emissions").show(5, False)

assert raw.count() == 195, "Esperavam-se 195 países."
assert len(raw.columns) == 36, "Esperavam-se 36 colunas."
# Remover caracteres de quebra de linha da coluna year
raw = raw.withColumn(
    "year",
    F.trim(F.regexp_replace(F.col("year"), r"[\r\n]", ""))
)
raw.select("country", "year").show(5, truncate=False)
assert raw.filter(F.col("year").isNull() | (F.trim(F.col("year")) != "2017")).count() == 0

(raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BRONZE_PATH))

print("Bronze Delta criada:", BRONZE_PATH)

Registos: 195
Colunas: 36
+-----------+------+-----------------+------------+-------------+
|country    |year  |unemployment_rate|gdp         |co2_emissions|
+-----------+------+-----------------+------------+-------------+
|Afghanistan|2017\r|11.2%            |18753000000 |4780         |
|Albania    |2017\r|13.6%            |13258000000 |5140         |
|Algeria    |2017\r|12.0%            |189881000000|158340       |
|Andorra    |2017\r|null             |3000000000  |471          |
|Angola     |2017\r|16.6%            |84377000000 |24250        |
+-----------+------+-----------------+------------+-------------+
only showing top 5 rows

+-----------+----+
|country    |year|
+-----------+----+
|Afghanistan|2017|
|Albania    |2017|
|Algeria    |2017|
|Andorra    |2017|
|Angola     |2017|
+-----------+----+
only showing top 5 rows

Bronze Delta criada: hdfs://hdfs-nn:9000/projeto/bronze/world_data_2017


In [8]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

necessarias = ["country", "year", "unemployment_rate", "gdp", "population", "co2_emissions"]
faltam = [c for c in necessarias if c not in bronze_df.columns]
assert not faltam, f"Colunas em falta: {faltam}"

def numero(nome_coluna, tipo):
    texto = F.trim(F.col(nome_coluna).cast("string"))
    limpo = F.regexp_replace(texto, r"[^0-9.\-]", "")
    vazio = texto.isNull() | (texto == "") | (limpo == "") | (limpo == "-")
    return F.when(vazio, F.lit(None).cast(tipo)).otherwise(limpo.cast(tipo))

df_clean = (
    bronze_df.select(
        F.trim(F.col("country")).alias("pais"),
        numero("year", "int").alias("ano"),
        numero("unemployment_rate", DecimalType(5, 1)).alias("taxa_desemprego_pct"),
        numero("gdp", DecimalType(38, 0)).alias("pib_usd"),
        numero("population", "long").alias("populacao"),
        numero("co2_emissions", "long").alias("emissoes_co2_kt"),
    )
    .withColumn("pais", F.when(F.col("pais") == "", None).otherwise(F.col("pais")))
)

# Compara campos preenchidos na Bronze com os resultados das conversões.
mapa = [
    ("year", "ano"),
    ("unemployment_rate", "taxa_desemprego_pct"),
    ("gdp", "pib_usd"),
    ("population", "populacao"),
    ("co2_emissions", "emissoes_co2_kt"),
]

# A validação direta das conversões é feita antes da seleção final, usando a própria expressão.
for entrada, saida in mapa:
    tipo = {
        "ano": "int", "taxa_desemprego_pct": DecimalType(5, 1),
        "pib_usd": DecimalType(38, 0), "populacao": "long", "emissoes_co2_kt": "long"
    }[saida]
    valor = F.trim(F.col(entrada).cast("string"))
    erros = bronze_df.filter(
        F.col(entrada).isNotNull() & (valor != "") & numero(entrada, tipo).isNull()
    ).count()
    print(f"Conversões inválidas ({entrada}):", erros)
    assert erros == 0, f"Conversões inválidas em {entrada}."

df_final = df_clean.withColumn(
    "flag_desemprego_em_falta", F.col("taxa_desemprego_pct").isNull()
)

print("=== Esquema Silver ===")
df_final.printSchema()
df_final.show(10, False)

Conversões inválidas (year): 0
Conversões inválidas (unemployment_rate): 0
Conversões inválidas (gdp): 0
Conversões inválidas (population): 0
Conversões inválidas (co2_emissions): 0
=== Esquema Silver ===
root
 |-- pais: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- taxa_desemprego_pct: decimal(5,1) (nullable = true)
 |-- pib_usd: decimal(38,0) (nullable = true)
 |-- populacao: long (nullable = true)
 |-- emissoes_co2_kt: long (nullable = true)
 |-- flag_desemprego_em_falta: boolean (nullable = false)

+-------------------+----+-------------------+-------------+---------+---------------+------------------------+
|pais               |ano |taxa_desemprego_pct|pib_usd      |populacao|emissoes_co2_kt|flag_desemprego_em_falta|
+-------------------+----+-------------------+-------------+---------+---------------+------------------------+
|Afghanistan        |2017|11.2               |18753000000  |37200000 |4780           |false                   |
|Albania            |201

In [9]:
n_bronze = bronze_df.count()
n_silver = df_final.count()

print("Bronze:", n_bronze, "| Silver:", n_silver)
print("Nulos por coluna:")
df_final.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_final.columns
]).show(truncate=False)

num_duplicados = df_final.groupBy("pais").count().filter("count > 1").count()
num_paises_nulos = df_final.filter(F.col("pais").isNull()).count()
num_taxas_invalidas = df_final.filter(
    (F.col("taxa_desemprego_pct") < 0) | (F.col("taxa_desemprego_pct") > 100)
).count()
num_anos_invalidos = df_final.filter(F.col("ano").isNull() | (F.col("ano") != 2017)).count()

print("Países duplicados:", num_duplicados)
print("Países sem nome:", num_paises_nulos)
print("Taxas inválidas:", num_taxas_invalidas)
print("Anos inválidos:", num_anos_invalidos)
print("Desemprego em falta:", df_final.filter("flag_desemprego_em_falta").count())

assert n_bronze == n_silver == 195, "O número de países mudou."
assert num_duplicados == 0, "Existem países repetidos."
assert num_paises_nulos == 0, "Existem países sem nome."
assert num_taxas_invalidas == 0, "Existem taxas fora do intervalo [0,100]."
assert num_anos_invalidos == 0, "Existem anos diferentes de 2017."

print("Validação: OK")

Bronze: 195 | Silver: 195
Nulos por coluna:
+----+---+-------------------+-------+---------+---------------+------------------------+
|pais|ano|taxa_desemprego_pct|pib_usd|populacao|emissoes_co2_kt|flag_desemprego_em_falta|
+----+---+-------------------+-------+---------+---------------+------------------------+
|0   |0  |16                 |4      |4        |3              |0                       |
+----+---+-------------------+-------+---------+---------------+------------------------+

Países duplicados: 0
Países sem nome: 0
Taxas inválidas: 0
Anos inválidos: 0
Desemprego em falta: 16
Validação: OK


In [10]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS silver LOCATION '{warehouse_location}/silver.db'")

log_path = spark._jvm.org.apache.hadoop.fs.Path(SILVER_PATH + "/_delta_log")
log_fs = log_path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())
ja_delta = log_fs.exists(log_path)

# Atualizar apenas o registo do Hive correspondente a esta tabela.
spark.sql(f"DROP TABLE IF EXISTS {TABELA_HIVE}")

if not ja_delta:
    print("Primeira execução: escrever Parquet e registar Hive...")
    (df_final.write
        .format("parquet")
        .mode("overwrite")
        .save(SILVER_PATH))
    spark.sql(f"""
        CREATE EXTERNAL TABLE {TABELA_HIVE} (
            pais STRING,
            ano INT,
            taxa_desemprego_pct DECIMAL(5,1),
            pib_usd DECIMAL(38,0),
            populacao BIGINT,
            emissoes_co2_kt BIGINT,
            flag_desemprego_em_falta BOOLEAN
        )
        STORED AS PARQUET
        LOCATION '{SILVER_PATH}'
    """)
    print("Parquet e tabela Hive criados.")
else:
    print("A tabela já é Delta; a atualizar diretamente...")
    (df_final.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(SILVER_PATH))

print("Silver:", SILVER_PATH)

Primeira execução: escrever Parquet e registar Hive...
Parquet e tabela Hive criados.
Silver: hdfs://hdfs-nn:9000/warehouse/silver.db/world_data_2017_silver


In [11]:
if not ja_delta:
    print("A converter Parquet para Delta...")
    spark.sql(f"CONVERT TO DELTA parquet.`{SILVER_PATH}`")
    print("Conversão concluída.")
else:
    print("A tabela já está em Delta.")

spark.sql(f"DROP TABLE IF EXISTS {TABELA_HIVE}")
spark.sql(f"CREATE TABLE {TABELA_HIVE} USING DELTA LOCATION '{SILVER_PATH}'")
print("Hive Delta disponível:", TABELA_HIVE)

A converter Parquet para Delta...
Conversão concluída.
Hive Delta disponível: silver.world_data_2017_silver


In [12]:
silver_check = spark.read.format("delta").load(SILVER_PATH)
print("Registos Silver:", silver_check.count())
print("Colunas Silver:", len(silver_check.columns))
silver_check.printSchema()

print("\nBrasil (World Data 2017):")
spark.sql(f"""
    SELECT pais, ano, taxa_desemprego_pct, pib_usd, populacao
    FROM {TABELA_HIVE}
    WHERE pais = 'Brazil'
""").show(truncate=False)

print("\nHistórico Delta:")
spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`").show(5, truncate=False)

assert silver_check.count() == 195
assert len(silver_check.columns) == 7
assert spark.sql(f"SELECT COUNT(*) AS total FROM {TABELA_HIVE}").first()["total"] == 195
print("Silver 2017 verificada com sucesso.")

Registos Silver: 195
Colunas Silver: 7
root
 |-- pais: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- taxa_desemprego_pct: decimal(5,1) (nullable = true)
 |-- pib_usd: decimal(38,0) (nullable = true)
 |-- populacao: long (nullable = true)
 |-- emissoes_co2_kt: long (nullable = true)
 |-- flag_desemprego_em_falta: boolean (nullable = true)


Brasil (World Data 2017):
+------+----+-------------------+-------------+---------+
|pais  |ano |taxa_desemprego_pct|pib_usd      |populacao|
+------+----+-------------------+-------------+---------+
|Brazil|2017|12.8               |2063515000000|209000000|
+------+----+-------------------+-------------+---------+


Histórico Delta:
+-------+-----------------------+------+--------+---------+-----------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+------------------------+------------+-----------------------------------+
|version|timestamp  

In [ ]:
from ydata_profiling import ProfileReport

world_pd = silver_check.toPandas()
profile = ProfileReport(
    world_pd,
    title="World Data 2017 - Data Profiling Silver",
    minimal=True
)
profile.to_file(HTML_PATH)
print("Relatório HTML criado:", HTML_PATH)

## Resumo para apresentar ao professor

1. Lemos o CSV de 2017 e copiámo-lo para a camada **RAW do HDFS**.
2. Criámos a **Bronze Delta** mantendo todos os campos originais como texto.
3. Na **Silver**, selecionámos sete colunas, convertendo texto numérico para tipos adequados sem acrescentar arredondamentos.
4. Confirmámos a qualidade com validações de linhas, tipos, duplicados e valores em falta.
5. Guardámos a Silver em **Parquet**, registámo-la no **Hive** e executámos `CONVERT TO DELTA`.
6. Confirmámos a leitura da tabela e gerámos o HTML de Data Profiling.

**Na Gold:** relacionar `pais` com a Global Superstore, filtrando a Superstore para `ano = 2017` quando fizeres a análise referente ao ano de 2017.